# Build a two-layer neural network

**Problem:** classify five input features into labels $\{0,1,2,3\}$ using a **5 → 10 → 4** network. Use ReLU in the hidden layer, softmax at the output, mean cross entropy as the loss, and Adam to train.

We will build forward and backward passes in NumPy, then compare learning from clean labels with memorizing a tiny noisy dataset. There is no autodiff framework.

Run the cells in order. Everything runs in your browser with JupyterLite; the Plotly charts are interactive. Examples are rows, and training uses float32.

In [ ]:
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['plotly==6.3.1', 'nbformat==5.10.4'])

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = 'plotly_mimetype'
pio.templates.default = 'plotly_white'
plot_config = {'responsive': False, 'displaylogo': False, 'scrollZoom': True}
class_colors = ['#087f72', '#c24a69', '#a96c08', '#386cb0']
input_dim, hidden_dim, output_dim = 5, 10, 4
classes = np.arange(output_dim)
print('Architecture:', input_dim, '-> ReLU', hidden_dim, '-> softmax', output_dim)

## 1. Make a small classification dataset

Generate four overlapping clusters with five features: **320 training examples** and **160 validation examples**, balanced across the four classes.

Split first, then standardize both sets using the **training mean and standard deviation only**. Validation data must not influence preprocessing or weight updates.

The chart shows the first two features; the network sees all five.

In [ ]:
data_rng = np.random.default_rng(17)
centers = np.array([[-1.5, -1.5, 1, 0, -1], [-1.5, 1.5, -1, 1, 0],
                    [1.5, -1.5, 0, -1, 1], [1.5, 1.5, 1, 1, 1]], dtype=np.float32)
train_parts, validation_parts = [], []
for center in centers:
    samples = (center + data_rng.normal(0, 0.95, size=(120, input_dim))).astype(np.float32)
    indices = data_rng.permutation(len(samples))
    train_parts.append(samples[indices[:80]])
    validation_parts.append(samples[indices[80:]])
raw_train = np.concatenate(train_parts)
raw_validation = np.concatenate(validation_parts)
y_train = np.repeat(classes, 80)
y_validation = np.repeat(classes, 40)
train_mean = raw_train.mean(axis=0)
train_scale = raw_train.std(axis=0)
X_train = ((raw_train - train_mean) / train_scale).astype(np.float32)
X_validation = ((raw_validation - train_mean) / train_scale).astype(np.float32)
assert X_train.shape == (320, 5) and X_validation.shape == (160, 5)
assert y_train.shape == (320,) and set(y_train) == {0, 1, 2, 3}
assert np.all(train_scale > 0)

projection = go.Figure()
for label, color in zip(classes, class_colors):
    selected = y_train == label
    projection.add_trace(go.Scatter(x=X_train[selected, 0], y=X_train[selected, 1], mode='markers',
        name=f'Class {label}', marker={'color': color, 'size': 6, 'opacity': 0.65}))
projection.update_layout(title={'text': 'Training data: 2D projection', 'font': {'size': 16}},
    height=420, margin={'l': 45, 'r': 15, 't': 55, 'b': 95},
    xaxis_title='Standardized feature 1', yaxis_title='Feature 2', legend={'orientation': 'h', 'y': -0.25})
projection.show(config=plot_config)
print('Train:', X_train.shape, 'Validation:', X_validation.shape)

## 2. Initialize the weights

| Parameter | Shape |
|---|---|
| $W_1$ | $5\times10$ |
| $b_1$ | $10$ |
| $W_2$ | $10\times4$ |
| $b_2$ | $4$ |

That is **104 trainable scalars**. Random weights let hidden neurons learn different features; biases start at zero. Scale hidden weights by $\sqrt{2/5}$ for ReLU and output weights by $\sqrt{1/10}$ to keep initial activations reasonably sized.

A seed makes the starting point reproducible.

In [ ]:
def initialize(seed=23, dtype=np.float32):
    rng = np.random.default_rng(seed)
    return {
        'W1': (rng.normal(size=(input_dim, hidden_dim)) * np.sqrt(2 / input_dim)).astype(dtype),
        'b1': np.zeros(hidden_dim, dtype=dtype),
        'W2': (rng.normal(size=(hidden_dim, output_dim)) / np.sqrt(hidden_dim)).astype(dtype),
        'b2': np.zeros(output_dim, dtype=dtype),
    }


parameters = initialize()
print({name: value.shape for name, value in parameters.items()})

## 3. Forward pass and loss

For a batch $X$ of shape $N\times5$:

$$Z_1=XW_1+b_1,\qquad H=\max(0,Z_1)$$
$$Z_2=HW_2+b_2.$$

Biases broadcast across rows. ReLU removes negative activations. The hidden array $H$ has shape $N\times10$; output scores (logits) $Z_2$ have shape $N\times4$.

Softmax turns each row of scores into class probabilities. Subtract the row maximum to avoid overflow:

$$s_{ik}=z_{ik}-\max_j z_{ij}$$
$$\log p_{ik}=s_{ik}-\log\sum_j e^{s_{ij}}.$$

Exponentiate to get $P$. For integer label $y_i$, select that class's log-probability and average:

$$L=-\frac1N\sum_i\log p_{i,y_i}.$$

Compute log-probabilities directly, since `log(softmax(...))` can take the log of an underflowed zero. Uniform predictions have loss $\log4\approx1.386$; confident correct predictions approach 0. Cache the arrays needed for backward.

In [ ]:
def softmax_cross_entropy(logits, labels):
    shifted = logits - logits.max(axis=1, keepdims=True)
    log_probabilities = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    probabilities = np.exp(log_probabilities)
    loss = -log_probabilities[np.arange(len(labels)), labels].mean()
    return probabilities, loss


def forward(inputs, labels, params):
    preactivation = inputs @ params['W1'] + params['b1']
    hidden = np.maximum(preactivation, 0)
    logits = hidden @ params['W2'] + params['b2']
    probabilities, loss = softmax_cross_entropy(logits, labels)
    cache = {'X': inputs, 'Z1': preactivation, 'H': hidden, 'P': probabilities}
    return loss, cache


preview_indices = np.array([0, 80, 160, 240])
initial_loss, initial_cache = forward(X_train[preview_indices], y_train[preview_indices], parameters)
print('One example per class:')
print(np.round(initial_cache['P'], 3))
print(f'Mean loss: {initial_loss:.3f}; uniform baseline: {np.log(4):.3f}')

## 4. Start backward at the loss

For one example, let $Y$ be its one-hot target: 1 at the true class, 0 elsewhere. Cross entropy gives

$$\frac{\partial\ell}{\partial p_k}=-\frac{Y_k}{p_k}.$$

The softmax Jacobian is

$$\frac{\partial p_k}{\partial z_j}=p_k(\delta_{kj}-p_j),$$

where $\delta_{kj}$ is 1 when $k=j$, otherwise 0. Raising one score increases its own probability and reduces the others.

Apply the chain rule; the $p_k$ terms cancel:

$$\frac{\partial\ell}{\partial z_j}=\sum_k-Y_k(\delta_{kj}-p_j)=p_j-Y_j.$$

For the **batch-mean** loss:

$$G_2=\frac{\partial L}{\partial Z_2}=\frac{P-Y}{N}.$$

In code: copy the probabilities, subtract 1 at each true class, and divide by batch size. There is no need to construct a Jacobian or divide by tiny probabilities.

## 5. Backward through the layers

For an affine layer $Z=AW+b$ with upstream gradient $G$, the weight gradient is $A^TG$, the input gradient is $GW^T$, and the bias gradient sums over rows because every example shares the bias.

**Output layer:**

$$\nabla_{W_2}L=H^TG_2,\qquad\nabla_{b_2}L=\sum_iG_{2,i}$$
$$G_H=G_2W_2^T.$$

**ReLU:** pass the gradient through positive activations; block it elsewhere. At zero, choose derivative 0.

$$G_1=G_H\odot\mathbf1[Z_1>0].$$

**Input layer:**

$$\nabla_{W_1}L=X^TG_1,\qquad\nabla_{b_1}L=\sum_iG_{1,i}$$
$$\nabla_XL=G_1W_1^T.$$

Each gradient has the shape of its parameter. Average **only once**, in $G_2$, and compute all gradients before changing any weights.

In [ ]:
def backward(cache, labels, params):
    logit_gradient = cache['P'].copy()
    logit_gradient[np.arange(len(labels)), labels] -= 1
    logit_gradient /= len(labels)

    weight2_gradient = cache['H'].T @ logit_gradient
    bias2_gradient = logit_gradient.sum(axis=0)
    hidden_gradient = logit_gradient @ params['W2'].T
    preactivation_gradient = hidden_gradient * (cache['Z1'] > 0)

    weight1_gradient = cache['X'].T @ preactivation_gradient
    bias1_gradient = preactivation_gradient.sum(axis=0)
    input_gradient = preactivation_gradient @ params['W1'].T
    gradients = {'W1': weight1_gradient, 'b1': bias1_gradient,
                 'W2': weight2_gradient, 'b2': bias2_gradient}
    return gradients, input_gradient


gradients, _ = backward(initial_cache, y_train[preview_indices], parameters)
print({name: value.shape for name, value in gradients.items()})

## 6. Update with Adam

Adam smooths gradients with a moving average $m$ and scales steps using a moving average of squared gradients $v$. Both start at zero:

$$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$$
$$v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.$$

Correct their initial bias toward zero:

$$\widehat m_t=\frac{m_t}{1-\beta_1^t},\qquad\widehat v_t=\frac{v_t}{1-\beta_2^t}.$$

Update each parameter elementwise:

$$\theta\leftarrow\theta-\eta\frac{\widehat m_t}{\sqrt{\widehat v_t}+\epsilon}.$$

Use $\eta=0.01$, $\beta_1=0.9$, $\beta_2=0.999$, and $\epsilon=10^{-8}$. Increment $t$ once per minibatch, not per parameter. Keep the moments between updates; reset them for each new training experiment.

In [ ]:
def adam_state(params):
    return {'t': 0, 'm': {name: np.zeros_like(value) for name, value in params.items()},
            'v': {name: np.zeros_like(value) for name, value in params.items()}}


def adam_step(params, gradients, state, learning_rate=0.01, beta1=0.9, beta2=0.999, adam_epsilon=1e-8):
    state['t'] += 1
    for name in params:
        gradient = gradients[name]
        state['m'][name] *= beta1
        state['m'][name] += (1 - beta1) * gradient
        state['v'][name] *= beta2
        state['v'][name] += (1 - beta2) * gradient**2
        corrected_mean = state['m'][name] / (1 - beta1**state['t'])
        corrected_variance = state['v'][name] / (1 - beta2**state['t'])
        params[name] -= learning_rate * corrected_mean / (np.sqrt(corrected_variance) + adam_epsilon)

## 7. Train on clean labels

The loop is **forward → backward → Adam** for each shuffled minibatch. A smaller last batch uses its actual size when averaging gradients.

Before training and after each epoch, measure full training and validation loss and accuracy. Predict the class with the largest probability: `probabilities.argmax(axis=1)`. Evaluation never updates weights.

The plotting helper shows loss and accuracy together; the dotted line is 25% chance accuracy. Each call to `train` starts with fresh weights and Adam state.

In [ ]:
def metrics(inputs, labels, params):
    loss, cache = forward(inputs, labels, params)
    accuracy = np.mean(cache['P'].argmax(axis=1) == labels)
    return float(loss), float(accuracy)


def train(inputs, labels, validation_inputs, validation_labels, epochs=120, batch_size=32, learning_rate=0.01, seed=23):
    params = initialize(seed)
    state = adam_state(params)
    rng = np.random.default_rng(seed + 1)
    history = {'epoch': [], 'train_loss': [], 'validation_loss': [], 'train_accuracy': [], 'validation_accuracy': []}
    for epoch in range(epochs + 1):
        if epoch > 0:
            order = rng.permutation(len(inputs))
            for start in range(0, len(inputs), batch_size):
                indices = order[start:start + batch_size]
                _, cache = forward(inputs[indices], labels[indices], params)
                gradients, _ = backward(cache, labels[indices], params)
                adam_step(params, gradients, state, learning_rate=learning_rate)
        training_loss, training_accuracy = metrics(inputs, labels, params)
        validation_loss, validation_accuracy = metrics(validation_inputs, validation_labels, params)
        history['epoch'].append(epoch)
        history['train_loss'].append(training_loss)
        history['validation_loss'].append(validation_loss)
        history['train_accuracy'].append(training_accuracy)
        history['validation_accuracy'].append(validation_accuracy)
    assert state['t'] == epochs * ((len(inputs) + batch_size - 1) // batch_size)
    assert all(np.isfinite(value).all() and value.dtype == np.float32 for value in params.values())
    return params, history, state


def learning_curves(history, title):
    figure = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.14,
                           subplot_titles=['Cross entropy', 'Accuracy'])
    for split, color in [('train', '#087f72'), ('validation', '#c24a69')]:
        for row, measure in [(1, 'loss'), (2, 'accuracy')]:
            figure.add_trace(go.Scatter(x=history['epoch'], y=history[f'{split}_{measure}'],
                name=split.title(), legendgroup=split, showlegend=(row == 1), line={'color': color}), row=row, col=1)
    figure.add_hline(y=0.25, line_dash='dot', line_color='#777777', row=2, col=1)
    figure.update_yaxes(range=[0, 1.05], tickformat='.0%', row=2, col=1)
    figure.update_xaxes(title_text='Epoch', row=2, col=1)
    figure.update_layout(title={'text': title, 'font': {'size': 16}}, height=620,
        margin={'l': 50, 'r': 15, 't': 85, 'b': 90}, hovermode='x unified', legend={'orientation': 'h', 'y': -0.17})
    figure.show(config=plot_config)


def report(history):
    for index, stage in [(0, 'Initial'), (-1, 'Final')]:
        print(stage)
        for split in ['train', 'validation']:
            print(f"  {split}: loss={history[f'{split}_loss'][index]:.4f}, accuracy={history[f'{split}_accuracy'][index]:.1%}")

In [ ]:
clean_epochs = 120
learning_rate = 0.01
batch_size = 32
clean_params, clean_history, clean_state = train(X_train, y_train, X_validation, y_validation,
    epochs=clean_epochs, batch_size=batch_size, learning_rate=learning_rate, seed=23)
report(clean_history)
learning_curves(clean_history, 'Learning from clean labels')

## 8. Show overfitting

Keep the same 5 → 10 → 4 network, but train on just **16 examples** with randomly reassigned, balanced labels. Reset weights and Adam, then train longer without regularization. Validation keeps its original labels.

This is a deliberate memorization experiment. High training accuracy with poor validation performance means the model fitted those examples without learning a useful rule. Some reassigned labels match by chance; the next cell reports how many.

Compare these curves with clean training. Cross entropy also measures confidence, so confidently wrong predictions can make validation loss rise even when accuracy barely changes.

In [ ]:
examples_per_class = 4
memorization_epochs = 1500
memorization_seed = 41
subset_rng = np.random.default_rng(memorization_seed)
subset_indices = np.concatenate([subset_rng.choice(np.flatnonzero(y_train == label), size=examples_per_class, replace=False) for label in classes])
X_tiny = X_train[subset_indices].copy()
y_tiny = subset_rng.permutation(np.repeat(classes, examples_per_class))
assert np.array_equal(np.bincount(y_tiny, minlength=4), np.full(4, examples_per_class))
print(f'{len(y_tiny)} training examples; {np.mean(y_tiny == y_train[subset_indices]):.1%} of reassigned labels match the originals.')
memorized_params, memorized_history, memorized_state = train(X_tiny, y_tiny, X_validation, y_validation,
    epochs=memorization_epochs, batch_size=len(y_tiny), learning_rate=learning_rate, seed=memorization_seed)
report(memorized_history)
print(f"Final accuracy gap: {memorized_history['train_accuracy'][-1] - memorized_history['validation_accuracy'][-1]:.1%}")
learning_curves(memorized_history, 'Memorizing 16 noisy labels')

## What to take away

- **Forward:** affine → ReLU → affine → softmax; cross entropy scores the true class.
- **Backward:** start with $(P-Y)/N$, apply matrix-product gradients, and mask with ReLU.
- **Training:** Adam reduces training loss. Validation tells us whether that helped on unseen examples.

Try changing the learning rate to 0.001 or 0.1, or increasing the noisy subset to eight examples per class. Rerun the training cells to start each experiment fresh.

The accompanying tests check all 104 parameter gradients and input gradients against float64 finite differences, plus softmax stability and Adam updates. Those checks stay outside this walkthrough.

Validation is for comparison, not a final performance claim; that would require a separate untouched test set.